# Lab 03 - Building a Tiny Transformer Language Model
**Module:** PUSL3205 Large Language Models and Generative AI

**Plan:** load the TinyStories dataset, build a small GPT-style Transformer from scratch in PyTorch, train it to predict the next character, measure loss and perplexity, generate stories from "Once upon a time", and see how the number of layers, heads and training steps changes the results.

Run on Google Colab with **Runtime > Change runtime type > T4 GPU**. Run the cells from top to bottom. Total time is roughly 30 minutes.

In [ ]:
!pip install -q datasets

In [ ]:
import math
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("Device:", device)
if use_amp:
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load and explore the dataset
TinyStories is a collection of very short stories written with a small child's vocabulary, so a small model can learn it. Streaming mode reads only the stories we ask for instead of downloading the whole 2 GB training file.

In [ ]:
from datasets import load_dataset

N_TRAIN, N_VAL = 20000, 1000
train_stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
val_stream = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)

train_stories = [row["text"] for row in train_stream.take(N_TRAIN)]
val_stories = [row["text"] for row in val_stream.take(N_VAL)]
print(len(train_stories), "training stories,", len(val_stories), "validation stories")

In [ ]:
for i in range(3):
    print(f"--- story {i} ---")
    print(train_stories[i], "\n")

In [ ]:
lengths = pd.DataFrame({
    "characters": [len(s) for s in train_stories],
    "words": [len(s.split()) for s in train_stories],
})
print(lengths.describe().round(0))

openings = pd.Series([" ".join(s.split()[:4]) for s in train_stories]).value_counts().head(5)
print("\nMost common story openings:\n", openings)
print("\nStories that contain a line break:", sum("\n" in s for s in train_stories))

lengths["words"].hist(bins=50)
plt.xlabel("Words per story")
plt.ylabel("Number of stories")
plt.title("Story length")
plt.show()

**Observations to write down:** how long are the stories on average, what do the openings look like, how simple is the vocabulary, and does every story have the same structure (one field called `text`, a beginning, a problem and an ending)?

## 2. Preprocess: character-level tokens
Each character becomes one token. That keeps the vocabulary tiny (about 100 symbols) and needs no tokenizer. The cost is that the model must learn spelling as well as grammar, and each token carries very little meaning.

Cleaning: curly quotes and dashes are replaced with plain ones and anything that is not ASCII is dropped, so rare symbols do not bloat the vocabulary. A special end-of-story character separates stories.

In [ ]:
REPLACE = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
           "\u2013": "-", "\u2014": "-", "\u2026": "..."}
EOS = "\x03"

def clean(story):
    for old, new in REPLACE.items():
        story = story.replace(old, new)
    return story.encode("ascii", "ignore").decode("ascii").strip()

train_text = EOS.join(clean(s) for s in train_stories) + EOS
val_text = EOS.join(clean(s) for s in val_stories) + EOS

chars = sorted(set(train_text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

def encode(text):
    return [stoi[c] for c in text if c in stoi]

def decode(ids):
    return "".join(itos[i] for i in ids)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Characters:", repr("".join(chars)))
print(f"Training tokens: {len(train_data):,} | validation tokens: {len(val_data):,}")
print(encode("Once upon a time"), "->", decode(encode("Once upon a time")))

## 3. Prepare training data: inputs and next-character targets
The model reads a window of `BLOCK_SIZE` characters and must predict the next character at **every position**. So the target is the input shifted one step to the right.

In [ ]:
BLOCK_SIZE = 256
BATCH_SIZE = 64

def get_batch(split):
    data = train_data if split == "train" else val_data
    starts = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in starts])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("Input batch shape:", tuple(xb.shape), "| target batch shape:", tuple(yb.shape))
for t in range(5):
    context = decode(xb[0][:t + 1].tolist())
    print(f"after {context!r} -> predict {decode([yb[0][t].item()])!r}")

## 4. Build the Transformer
Four parts, each a small class:
- **Self-attention:** every character looks at the characters before it and decides which ones matter. The **mask** blocks looking at future characters, because the model must not see the answer. Several **heads** run in parallel, each free to focus on something different.
- **Feed-forward layer:** a small two-layer network applied to each position after attention.
- **Block:** attention then feed-forward, each with **layer normalisation** before it and a **residual connection** (the input is added back to the output) after it.
- **TinyGPT:** token embeddings plus position embeddings, a stack of blocks, a final layer norm, and a linear layer that outputs a score for every character in the vocabulary.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx

## 5. Initialise the model, the loss and the optimizer
Weights start as small random numbers. The loss is **cross-entropy**: how surprised the model is by the true next character. A model that guesses at random has a loss of about `ln(vocab_size)`, so the first printed loss should be close to that number. If it is not, something is wrong.

In [ ]:
CONFIG = dict(n_layer=6, n_head=6, n_embd=384, dropout=0.1)

def build_model(cfg):
    return TinyGPT(vocab_size, BLOCK_SIZE, **cfg).to(device)

model = build_model(CONFIG)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.2f} M")
print(f"Loss of random guessing, ln({vocab_size}) = {math.log(vocab_size):.3f}")

model.eval()
with torch.no_grad():
    _, first_loss = model(*get_batch("train"))
print(f"Loss of the untrained model: {first_loss.item():.3f}")

## 6. Train
Each step: take a batch, predict the next character at every position, compute the cross-entropy loss, backpropagate, and update the weights with **AdamW**. Extra tricks that help training: a **learning-rate schedule** (warm up, then decay), **gradient clipping** (stops one bad step from wrecking the model) and **mixed precision** (faster on the GPU).

In [ ]:
def get_lr(step, max_steps, lr=1e-3, min_lr=1e-4, warmup=100):
    if step < warmup:
        return lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * progress)) * (lr - min_lr)


@torch.no_grad()
def estimate_loss(model, eval_batches=30):
    model.eval()
    result = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(split)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                _, loss = model(x, y)
            losses.append(loss.item())
        result[split] = sum(losses) / len(losses)
    model.train()
    return result


def train(model, max_steps, eval_every=250, log=True):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history = []
    start = time.time()
    model.train()
    for step in range(max_steps + 1):
        if step % eval_every == 0 or step == max_steps:
            losses = estimate_loss(model)
            history.append({"step": step, "train": losses["train"], "val": losses["val"]})
            if log:
                print(f"step {step:5d} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f} | {time.time() - start:.0f}s")
        if step == max_steps:
            break

        for group in optimizer.param_groups:
            group["lr"] = get_lr(step, max_steps)

        x, y = get_batch("train")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            _, loss = model(x, y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

    return history, time.time() - start

In [ ]:
MAX_STEPS = 3000
epochs = MAX_STEPS * BATCH_SIZE * BLOCK_SIZE / len(train_data)
print(f"{MAX_STEPS} steps = about {epochs:.1f} passes over the training text")

history, train_seconds = train(model, MAX_STEPS)
print(f"\nTraining took {train_seconds / 60:.1f} minutes")

## 7. Monitor training: loss curves
The training loss is measured without dropout, on batches from the training text. The validation loss uses stories the model has never seen. If validation loss stops falling or rises while training loss keeps falling, the model is starting to memorise.

In [ ]:
hist = pd.DataFrame(history)
plt.plot(hist["step"], hist["train"], marker="o", label="train")
plt.plot(hist["step"], hist["val"], marker="o", label="validation")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Loss curves")
plt.legend()
plt.show()
hist.round(3)

## 8. Evaluate: validation loss and perplexity
**Perplexity** is `exp(loss)`. Read it as "the model is as unsure as if it were choosing between this many characters at each step". A perplexity of 1 would be perfect, and `vocab_size` is random guessing.

In [ ]:
final = estimate_loss(model, eval_batches=100)
val_perplexity = math.exp(final["val"])
print(f"Validation loss : {final['val']:.3f}")
print(f"Perplexity      : {val_perplexity:.2f}  (random guessing would be {vocab_size})")
print(f"Train loss      : {final['train']:.3f}")

## 9. Generate stories
Start from a prompt and let the model add one character at a time. **Temperature** controls randomness (low = safe and repetitive, high = creative and messy). **Top-k** only lets it choose among the k most likely characters.

In [ ]:
def write_story(model, prompt="Once upon a time", max_chars=500, temperature=0.8, top_k=40):
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_chars, temperature, top_k)[0].tolist()
    return decode(out).replace(EOS, "\n[END OF STORY]\n")

for temp in (0.5, 0.8, 1.2):
    print(f"===== temperature {temp} =====")
    print(write_story(model, "Once upon a time", 500, temp), "\n")

In [ ]:
print(write_story(model, "One day, a little girl named Mia", 500, 0.8))

## 10. Experiment with the settings
Each configuration is trained for the same number of steps (except the last one) with the same seed, so the differences come from the setting. Compare the number of layers, the number of heads (same parameter count, different split), and the number of training steps.

In [ ]:
EXPERIMENTS = [
    ("2 layers, 2 heads, 128 dim", dict(n_layer=2, n_head=2, n_embd=128, dropout=0.1), 1000),
    ("4 layers, 4 heads, 256 dim", dict(n_layer=4, n_head=4, n_embd=256, dropout=0.1), 1000),
    ("6 layers, 6 heads, 384 dim", dict(n_layer=6, n_head=6, n_embd=384, dropout=0.1), 1000),
    ("6 layers, 1 head, 384 dim", dict(n_layer=6, n_head=1, n_embd=384, dropout=0.1), 1000),
    ("6 layers, 6 heads, only 250 steps", dict(n_layer=6, n_head=6, n_embd=384, dropout=0.1), 250),
]

rows, samples = [], {}
for name, cfg, steps in EXPERIMENTS:
    torch.manual_seed(1337)
    m = build_model(cfg)
    hist_m, seconds = train(m, steps, eval_every=steps, log=False)
    val_loss = hist_m[-1]["val"]
    rows.append({
        "Setting": name,
        "Parameters (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 2),
        "Steps": steps,
        "Val loss": round(val_loss, 3),
        "Perplexity": round(math.exp(val_loss), 2),
        "Minutes": round(seconds / 60, 1),
    })
    torch.manual_seed(7)
    samples[name] = write_story(m, "Once upon a time", 250, 0.8)
    del m
    torch.cuda.empty_cache()
    print("done:", name)

experiment_table = pd.DataFrame(rows)
experiment_table

In [ ]:
for name, text in samples.items():
    print(f"===== {name} =====")
    print(text, "\n")

## 11. Save the model
The weights and the settings are saved separately, so the model can be rebuilt and used later without training again.

In [ ]:
import os

torch.save(model.state_dict(), "tinystories_gpt.pt")
with open("tinystories_gpt_config.json", "w") as f:
    json.dump({"config": CONFIG, "block_size": BLOCK_SIZE, "chars": chars}, f)

print(f"Saved weights: {os.path.getsize('tinystories_gpt.pt') / 1024**2:.1f} MB")

with open("tinystories_gpt_config.json") as f:
    saved = json.load(f)
reloaded = TinyGPT(len(saved["chars"]), saved["block_size"], **saved["config"]).to(device)
reloaded.load_state_dict(torch.load("tinystories_gpt.pt", map_location=device))
print(write_story(reloaded, "Once upon a time", 300, 0.8))

## Optional: a chat window with Gradio

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def story_ui(prompt, temperature, length):
    return write_story(model, prompt, int(length), temperature)

demo = gr.Interface(
    fn=story_ui,
    inputs=[
        gr.Textbox(value="Once upon a time", label="Start of the story"),
        gr.Slider(0.3, 1.5, value=0.8, label="Temperature"),
        gr.Slider(100, 800, value=400, step=50, label="Characters to write"),
    ],
    outputs=gr.Textbox(label="Story"),
    title="Tiny Story Generator",
)
demo.launch()

## Discussion: what makes the model perform better?
Pick four or five from this list that you can point to in the code above, and explain in your own words what each one does and why it helps:
- causal mask, positional embeddings, multi-head attention
- residual connections and layer normalisation
- feed-forward layers
- dropout, weight decay (AdamW)
- learning-rate warm-up and decay, gradient clipping
- mixed precision, weight tying, scaled weight initialisation
- temperature and top-k at generation time

Then answer from your own results: did more layers help, did more heads help when the parameter count stayed the same, what happened with only 250 steps, and what does the gap between training and validation loss say?